## Pseudo-bulk differential expression analysis. 
The method is based on the following publication: https://doi.org/10.1101/2024.08.03.606460 method section: subsection "Identification of genes and pathways changing with donor phenotypes." page 25. <br>
Here, the top 4000 most variable genes are used for gathering the pseudobulk dataset

In [2]:
import scanpy as sc
import pickle 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob

/home/frederik/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


### load in subclustering AnnData object from 'scanpy-subclustering.ipynb'

In [3]:
subcluster_paths = glob.glob('../../h5ad/subcluster*')
subcluster_names = [subcluster_path.replace('../../h5ad/subcluster_', '').replace('.h5ad', '') for subcluster_path in subcluster_paths]

subcluster = {}
for i, subcluster_path in enumerate(subcluster_paths):
    subcluster[subcluster_names[i]] = sc.read_h5ad(subcluster_path)

In [ ]:
# Adjust number of genes to include.
for subcluster_name, subcluster_adata in subcluster.items():
    sc.pp.highly_variable_genes(subcluster_adata, n_top_genes=4000)

### Getting pseudobulk objects.
1. define the genelist
2. for each cluster:
    * get the sum of all the observed counts per gene for all cells in a specific donor.
    * Concatenate all genes into a DataFrame, then convert into a AnnData
    * Assign the cluster-id is to AnnData.obs 
3. Concatenate all AnnData objects (i.e., 1 AnnData object represents a single cluster)

In [4]:
def pseduo_bulk_sum_per_cluster(adata, cluster_col, sample_col): 
    gene_list = adata.var_names[adata.var['highly_variable'] == True]
    pseudo_bulk_sum_per_cluster_adata = {}
    print(f'{len(gene_list)} genes will be included')
    for cluster in np.unique(adata.obs[cluster_col]):
        ## Fetch gene expression for each highly variable gene. 
        ## get the average expression per sample (pseudo-bulk)
        pseudo_bulk_sum = {}
        for gene in gene_list:
            geneexpr = sc.get.obs_df(adata[adata.obs['leiden'] == cluster], [sample_col, gene])
            pseudo_bulk_sum[gene] = geneexpr.groupby(sample_col, observed=False).sum()
        
        ## Concatenate gene sums into DataFrame
        pseudo_bulk_sum_df = pd.concat(pseudo_bulk_sum, axis=1)
        pseudo_bulk_sum_df.columns = pseudo_bulk_sum_df.columns.get_level_values(0) # Multiple index columns for some reason. Fix that.
        ## Change into AnnData
        to_adata = sc.AnnData(pseudo_bulk_sum_df)
        to_adata.obs['cluster'] = cluster
        pseudo_bulk_sum_per_cluster_adata[cluster] = to_adata
        
    return sc.concat(pseudo_bulk_sum_per_cluster_adata)

In [ ]:
# This takes long time to process (30-40 minutes)
pseudobulk = {}
for subcluster_name, subcluster_adata in subcluster.items():
    print(subcluster_name)
    pseudobulk[subcluster_name] = pseduo_bulk_sum_per_cluster(subcluster_adata, 'leiden', 'sample')

Acinar
4000 genes will be included


/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Ductal
4000 genes will be included


/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Endocrine
4000 genes will be included


...

/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Immune
4000 genes will be included


/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Stellate
4000 genes will be included


/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Differential expression analysis based on pseudobulk data

In [ ]:
for pseudobulk_name, pseudobulk_adata in pseudobulk.items():
    sc.pp.normalize_total(pseudobulk_adata)
    sc.tl.rank_genes_groups(pseudobulk_adata, groupby='cluster')

    DE = {}
    for cluster in np.unique(pseudobulk_adata.obs['cluster']):
        DE[cluster] = sc.get.rank_genes_groups_df(pseudobulk_adata, group=cluster)
        DE[cluster].to_csv(f'../../pseudo_{pseudobulk_name}_{cluster}.txt', sep = '\t')

Cluster 0: 125 DE genes
Cluster 1: 99 DE genes
Cluster 2: 208 DE genes
Cluster 3: 149 DE genes


In [ ]:
# Save to pickle file
with open('../../pseudobulk_subclusters.pickle', 'wb') as handle:
    pickle.dump(pseudobulk, handle, protocol=pickle.HIGHEST_PROTOCOL)

Further exploration of acinar sublcuster with different values of resolution

In [ ]:
acinar_3cl = subcluster['Acinar'].copy()
acinar_4cl = subcluster['Acinar'].copy()

In [ ]:
# This and next Code cell is then run for both acinar_3cl and acinar_4cl object
sc.tl.leiden(acinar_3cl, flavor='igraph', resolution=.4)
print(len(np.unique(acinar_3cl.obs['leiden'])))
acinar_3cl_pseudo_bulk = pseduo_bulk_sum_per_cluster(acinar_3cl, 'leiden', 'sample')

3
4000 genes will be included


/home/frederik/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
sc.pp.normalize_total(acinar_3cl_pseudo_bulk)
sc.tl.rank_genes_groups(acinar_3cl_pseudo_bulk, groupby='cluster')

DE = {}
for cluster in np.unique(acinar_3cl_pseudo_bulk.obs['cluster']):
    DE[cluster] = sc.get.rank_genes_groups_df(acinar_3cl_pseudo_bulk, group=cluster)
    DE[cluster].to_csv(f'../../pseudo_Acinar4cl_cluster{cluster}.txt', sep = '\t')

for cluster, df in DE.items():
    length = len(df['names'][(df['logfoldchanges'] > 0) & (df['pvals_adj'] < 0.1)])
    print(f'Cluster {cluster}: {length} DE genes')

NameError: name 'acinar_4cl_pseudo_bulk' is not defined

##